# Dense Coherence Reward Shaping for GPT-2 (125M) — Reference Implementation

PPO fine-tuning of GPT-2 Base, comparing a **sparse** terminal reward against a
**dense** per-step coherence reward from a RoBERTa Bradley-Terry model.

> Running this reproduces the **method and similar results**, not the exact per-seed
> numbers in the paper — PPO training is stochastic. Authoritative metrics are logged
> in Weights & Biases. The reward model is loaded from the Hugging Face Hub:
> https://huggingface.co/Paimagham/roberta-story-coherence

**Environment:** these pins target the original (mid-2025) training environment.
Newer Colab/Kaggle base images have moved past this stack; use the included
`Dockerfile` for a guaranteed clean run, or install the pins below and **restart the
runtime** before importing.

In [ ]:
# STEP 1 - GPU check
!nvidia-smi
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

In [ ]:
# STEP 2 - Install (pinned). After this cell, RESTART THE RUNTIME before continuing.
!pip install -q --force-reinstall --no-deps \
    torch==2.2.2 trl==0.8.6 transformers==4.40.0 accelerate==0.29.3 \
    huggingface-hub==0.22.2 tokenizers==0.19.1 safetensors==0.4.3 datasets==2.19.0
!pip install -q "numpy<2" pyarrow_hotfix scipy wandb sentencepiece
print("Installed - now Runtime > Restart session, then continue from STEP 3.")

In [ ]:
# STEP 2B - Patch TRL (numpy int64 fix + ratio clamp). Run after restart.
import trl.trainer.ppo_trainer as _m, inspect, importlib
p = inspect.getfile(_m); s = open(p).read()
a = 'mini_batch_inds = backward_batch_inds[mini_batch_start:mini_batch_end]'
b = 'mini_batch_inds = [int(x) for x in backward_batch_inds[mini_batch_start:mini_batch_end]]'
if a in s: s = s.replace(a, b); print('numpy int64 patch applied')
c = 'ratio = torch.exp(logprobs - old_logprobs)'
d = 'ratio = torch.exp(torch.clamp(logprobs - old_logprobs, -5.0, 5.0))'
if c in s: s = s.replace(c, d); print('ratio clamp patch applied')
open(p, 'w').write(s); importlib.reload(_m)
from trl import PPOTrainer
print('TRL ready')

In [ ]:
# STEP 3 - Config
import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEEDS = [42, 99, 7]
NUM_STEPS = 90
MAX_TOKENS = 64
MODEL = 'gpt2'
SPARSE_THRESHOLD = 1.0
ROBERTA_ID = "Paimagham/roberta-story-coherence"   # loads from Hugging Face
print('Device:', DEVICE, '| seeds', SEEDS, '| steps', NUM_STEPS)

In [ ]:
# STEP 4 - PPO config (final stable config from the paper)
from trl import PPOConfig
PPO_CONFIG = PPOConfig(
    model_name=MODEL, learning_rate=1e-6,
    batch_size=64, mini_batch_size=32, ppo_epochs=1,
    cliprange=0.1, cliprange_value=0.1, vf_coef=0.1,
    gamma=1.0, lam=0.95, init_kl_coef=0.5, target=6,
    horizon=10000, max_grad_norm=0.5,
)
print('PPO config ready')

In [ ]:
# STEP 5 - Load RoBERTa coherence reward model from Hugging Face + normalizer
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

reward_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_ID)
reward_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_ID).to(DEVICE).eval()
print('Reward model loaded from', ROBERTA_ID)

class RunningNormalizer:
    """Welford online mean/variance with baseline subtraction and clipping."""
    def __init__(self, clip=1.0, eps=1e-8):
        self.n = 0; self.mean = 0.0; self.M2 = 0.0; self.clip = clip; self.eps = eps
    def update(self, x):
        self.n += 1; d = x - self.mean; self.mean += d / self.n; self.M2 += d * (x - self.mean)
    @property
    def std(self):
        return (self.M2 / self.n) ** 0.5 if self.n > 1 else 1.0
    def normalize(self, x):
        self.update(x); z = (x - self.mean) / (self.std + self.eps)
        return float(np.clip(z, -self.clip, self.clip))

def get_roberta_score(text):
    """Continuous coherence score = logit[1] - logit[0]."""
    inp = reward_tokenizer(text, return_tensors='pt', truncation=True,
                           max_length=512, padding=True).to(DEVICE)
    with torch.no_grad():
        lg = reward_model(**inp).logits
    return float(lg[0][1].item() - lg[0][0].item())

# sanity check: coherent text should score higher than word-salad
print('coherent  :', round(get_roberta_score("The cat sat on the mat and fell asleep in the sun."), 3))
print('incoherent:', round(get_roberta_score("Cat the mat purple sky running yesterday of."), 3))

In [ ]:
# STEP 6 - Reward functions
def dense_reward(text, normalizer):
    """Continuous per-generation coherence reward, normalized and clipped."""
    return normalizer.normalize(get_roberta_score(text))

def sparse_reward(text):
    """Binary terminal reward: 1 if coherence score exceeds threshold, else 0."""
    return 1.0 if get_roberta_score(text) > SPARSE_THRESHOLD else 0.0

In [ ]:
# STEP 7 - Story prompts (extend/replace with your full prompt set)
PROMPT_TEMPLATES = [
    'Once upon a time there was a little girl',
    'The old lighthouse stood alone on the cliff',
    'It was the first day of school',
    'Deep in the forest lived a creature',
    'The rocket launched into the stars',
    'Every morning the baker woke before the sun',
    'The two friends had not met in years',
    'A small dog wandered into the village',
    'The library held many secrets',
    'Nobody believed her when she found a map',
    'The storm came without any warning',
    'He always wanted to learn the piano',
    'The market was busy with people',
    'She opened the box and found a letter',
    'The mountain path was steep but worth it',
    'Three children built the biggest sandcastle',
]
PROMPTS = ['write me a story starting with: ' + p for p in PROMPT_TEMPLATES]
print(len(PROMPTS), 'prompts')

In [ ]:
# STEP 8 - Training loop (one condition, one seed)
import random, json
from pathlib import Path
from transformers import GPT2TokenizerFast
from trl import PPOTrainer, AutoModelForCausalLMWithValueHead
from google.colab import drive

drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/DenseCoherence_Reconstruction'
for cond in ['dense', 'sparse']:
    for s in SEEDS:
        Path(f'{SAVE_DIR}/{cond}/seed_{s}').mkdir(parents=True, exist_ok=True)
Path(f'{SAVE_DIR}/plots').mkdir(parents=True, exist_ok=True)

tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

def set_seeds(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def train(condition, seed):
    assert condition in ('dense', 'sparse')
    print(f'\n=== {condition.upper()} seed {seed} ===')
    set_seeds(seed)
    policy = AutoModelForCausalLMWithValueHead.from_pretrained(MODEL).to(DEVICE)
    ref = AutoModelForCausalLMWithValueHead.from_pretrained(MODEL).to(DEVICE)
    for pr in ref.parameters(): pr.requires_grad = False
    trainer = PPOTrainer(config=PPO_CONFIG, model=policy, ref_model=ref, tokenizer=tokenizer)
    norm = RunningNormalizer(clip=1.0); metrics = []
    for step in range(NUM_STEPS):
        batch = [random.choice(PROMPTS) for _ in range(PPO_CONFIG.batch_size)]
        queries = [tokenizer.encode(pt, return_tensors='pt').squeeze(0).to(torch.long).to(DEVICE) for pt in batch]
        resp = trainer.generate(queries, max_new_tokens=MAX_TOKENS, min_new_tokens=10,
                                temperature=1.0, pad_token_id=tokenizer.eos_token_id)
        resp = [r.to(torch.long) for r in resp]
        rewards = []
        for q, r in zip(queries, resp):
            gen = tokenizer.decode(r[len(q):], skip_special_tokens=True)
            v = dense_reward(gen, norm) if condition == 'dense' else sparse_reward(gen)
            rewards.append(torch.tensor(v, dtype=torch.float32))
        st = trainer.step(queries, resp, rewards)
        metrics.append({'step': step,
            'kl_divergence': float(st.get('objective/kl', 0)),
            'value_loss': float(st.get('ppo/loss/value', 0)),
            'policy_loss': float(st.get('ppo/loss/policy', 0)),
            'entropy': float(st.get('ppo/policy/entropy', 0)),
            'episode_return': float(torch.stack(rewards).mean().item())})
        if step % 10 == 0:
            m = metrics[-1]
            print(f'  step {step:3d} KL={m["kl_divergence"]:.3f} VL={m["value_loss"]:.3f} RET={m["episode_return"]:.3f}')
    out = Path(SAVE_DIR) / condition / f'seed_{seed}'
    policy.save_pretrained(str(out / 'final'))
    json.dump(metrics, open(out / 'metrics.json', 'w'), indent=2)
    print(f'  saved -> {out}')
    return metrics

print('train() ready')

In [ ]:
# STEP 9 - Run all six conditions (2 rewards x 3 seeds). Long-running.
all_metrics = {}
for cond in ['dense', 'sparse']:
    for s in SEEDS:
        all_metrics[(cond, s)] = train(cond, s)
print('\nAll runs complete')

In [ ]:
# STEP 10 - Held-out evaluation (reward, diversity, repetition, win-rate)
from transformers import AutoModelForCausalLM

HELDOUT_PROMPTS = ['write me a story starting with: ' + p for p in [
    'The clock tower struck midnight',
    'A stranger arrived in the quiet town',
    'The garden had not been touched in years',
    'She heard a knock at the door',
    'The last train had already left',
    # extend to 50 prompts for a full evaluation
]]

def distinct_n(texts, n):
    total, uniq = 0, set()
    for t in texts:
        toks = t.split(); grams = list(zip(*[toks[i:] for i in range(n)]))
        total += len(grams); uniq.update(grams)
    return len(uniq) / max(total, 1)

def repetition_rate(text):
    toks = text.split()
    if len(toks) < 2: return 0.0
    return sum(1 for i in range(1, len(toks)) if toks[i] == toks[i-1]) / (len(toks) - 1)

def generate(model, prompt):
    ids = tokenizer.encode(prompt, return_tensors='pt').to(DEVICE)
    out = model.generate(ids, max_new_tokens=MAX_TOKENS, min_new_tokens=10,
                         do_sample=True, temperature=1.0, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

reference = AutoModelForCausalLM.from_pretrained('gpt2').to(DEVICE).eval()
dense_model = AutoModelForCausalLM.from_pretrained(f'{SAVE_DIR}/dense/seed_42/final').to(DEVICE).eval()

ref_texts, dense_texts, wins = [], [], 0
for pmt in HELDOUT_PROMPTS:
    rt, dt = generate(reference, pmt), generate(dense_model, pmt)
    ref_texts.append(rt); dense_texts.append(dt)
    if get_roberta_score(dt) > get_roberta_score(rt): wins += 1

ref_r = np.mean([get_roberta_score(t) for t in ref_texts])
dns_r = np.mean([get_roberta_score(t) for t in dense_texts])
print(f'Reference reward: {ref_r:.3f}')
print(f'Dense reward:     {dns_r:.3f}  (delta {dns_r-ref_r:+.3f})')
print(f'Dense wins: {wins}/{len(HELDOUT_PROMPTS)}')
print(f'Distinct-2  ref={distinct_n(ref_texts,2):.3f}  dense={distinct_n(dense_texts,2):.3f}')
print(f'Repetition  ref={np.mean([repetition_rate(t) for t in ref_texts]):.4f}  dense={np.mean([repetition_rate(t) for t in dense_texts]):.4f}')

In [ ]:
# STEP 11 - Six-panel training-dynamics plot (sparse vs dense)
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

KEYS = [('episode_return','Episode Return'), ('kl_divergence','KL Divergence'),
        ('value_loss','Value Loss'), ('entropy','Policy Entropy'), ('policy_loss','Policy Loss')]

def series(cond, k):
    a = [[r[k] for r in all_metrics[(cond, s)]] for s in SEEDS if (cond, s) in all_metrics]
    if not a: return None, None
    n = min(len(x) for x in a); M = np.array([x[:n] for x in a]); return M.mean(0), M.std(0)

fig, ax = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('PPO training dynamics: sparse vs. dense coherence reward (mean +/- 1 SD, seeds 42/99/7)',
             fontsize=13, fontweight='bold')
for a, (k, lbl) in zip(ax.flat, KEYS):
    for cond, col in [('sparse', '#C00000'), ('dense', '#2E74B5')]:
        m, sd = series(cond, k)
        if m is None: continue
        x = np.arange(len(m))
        a.fill_between(x, m - sd, m + sd, alpha=0.15, color=col)
        a.plot(x, gaussian_filter1d(m, 3), color=col, lw=2, label=cond)
    a.set_title(lbl, fontweight='bold'); a.set_xlabel('training step'); a.legend(); a.grid(alpha=0.2)
ax.flat[-1].axis('off')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/plots/all_metrics_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved')